In [1]:
!pip install trl bitsandbytes

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from google.colab import drive
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset, load_dataset
from trl import SFTConfig, SFTTrainer




drive.mount('/content/drive')

ModuleNotFoundError: No module named 'trl'

In [3]:
!hf download AI-MO/NuminaMath-CoT --repo-type=dataset

Fetching 8 files: 100% 8/8 [00:00<00:00, 95869.81it/s]
/root/.cache/huggingface/hub/datasets--AI-MO--NuminaMath-CoT/snapshots/9d8d210c9f6a36c8f3cd84045668c9b7800ef517


In [4]:
ds = load_dataset("AI-MO/NuminaMath-CoT")
train_ds, val_ds = train_test_split(ds['train'].to_pandas(), test_size=0.2)

train_ds = Dataset.from_pandas(train_ds)
val_ds = Dataset.from_pandas(val_ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['source', 'problem', 'solution', 'messages'],
        num_rows: 859494
    })
    test: Dataset({
        features: ['source', 'problem', 'solution', 'messages'],
        num_rows: 100
    })
})


In [11]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

#Use QLoRA with 4-bit precision to reduce ram usage. Is more efficient because we don't have to load the full model this way
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False
)

model = prepare_model_for_kbit_training(model)


#Use LoRA adapters so we don't have to fine tune the whole model, just smaller adapters
peft_config = LoraConfig(
    r=16,       # Rank (Higher = smarter but more VRAM. 16-64 is standard)
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
stf_config = SFTConfig(
    output_dir="./sft_results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    logging_steps=10,
    fp16=True,
)

sft_trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=stf_config
)


sft_trainer.train()
sft_trainer.save_model("sft_model")


Tokenizing train dataset:   0%|          | 0/687595 [00:00<?, ? examples/s]

Exception ignored in: <function _xla_gc_callback at 0x792ae5a43f60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


Would be great if you could fine tine this model and save it somewhere in the drive. Feel free to add a couple extra things to fine tune it if you want, up to you.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("sft_model", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("sft_model")

Below is something that should be able to give you accuracy for the fine-tuned model, idk if it will work or not though tbh since I haven't gotten the fine-tuned model.

In [ ]:
import torch

def generate_answer(question, max_new_tokens=128):
    prompt = f"<s>[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text

In [ ]:
def extract_response(generated_text):
    if "[/INST]" in generated_text:
        return generated_text.split("[/INST]")[-1].strip()
    return generated_text.strip()

In [ ]:
def evaluate_accuracy(test_ds):
    correct = 0

    for ex in test_ds:
        q = ex["question"]
        gold = str(ex["answer"]).strip()

        raw_output = generate_answer(q)
        pred = extract_response(raw_output)

        # normalize prediction
        pred_clean = pred.strip().lower()
        gold_clean = gold.strip().lower()

        if pred_clean == gold_clean:
            correct += 1

    return correct / len(test_ds)

In [ ]:
accuracy = evaluate_accuracy(ds["test"])
print(f"Accuracy: {accuracy * 100:.2f}%")